# 02 — Matched Ablation, Router Sensitivity, and Year-over-Year Analysis

This notebook reproduces the component analyses used in the revised manuscript.

The important distinction is that **structural/feature variants freeze the locally selected full-model setting** rather than receiving a new test-informed architecture. The sequential-only result is the primary mechanism estimand; a separately labeled rerouted result is secondary.

Annual-information variants are evaluated only for Cluster 1 and Cluster 2 because the one-year BDG series do not contain matched prior-year history.


## 1. Setup

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "42")

import lash_revision_core as core
import lash_revision_ablation as ablation
import lash_deadline48 as engine
from lash_revision_core import ExperimentConfig
from lash_hardware_optimized import apply_hardware_patch
from lash_per_dataset_hpo import (
    PerDatasetHPOPolicy,
    _read_existing_settings,
    run_per_dataset_controlled_mechanisms,
)

apply_hardware_patch()
policy = PerDatasetHPOPolicy(computational_budget_hours=48.0)

physical = os.cpu_count() or 1
config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    dataset_keys=("CLUSTER_1", "CLUSTER_2", "BDG_EDU", "BDG_DORM"),
    run_profile="paper",
    weather_mode="historical_only",
    hpo_seeds=policy.hpo_seeds,
    final_refit_seeds=policy.lash_final_seeds,
    primary_hpo_repeats=2,
    external_hpo_repeats=2,
    hpo_trials_per_dimension=1,
    hpo_min_trials=3,
    hpo_max_trials=5,
    max_epochs=policy.max_epochs,
    early_stopping_patience=policy.patience,
    tree_horizon_jobs=min(8, max(1, physical)),
    tree_threads_per_model=1,
    require_cuda=False,
    use_amp=True,
    save_models=True,
    resume=True,
)

for key in config.dataset_keys:
    settings = _read_existing_settings(config, key, policy)
    if settings.get("status") != "complete":
        raise RuntimeError(f"Notebook 01 must complete local HPO for {key} first.")
print("Frozen dataset-local settings verified for all four datasets.")


## 2. Router sensitivity on all four datasets

In [ ]:
router_results = ablation.run_router_sensitivity_suite(
    config,
    dataset_keys=config.dataset_keys,
)
for key, table in router_results.items():
    print("\n", key)
    display(table.head(20))


## 3. Controlled structural, feature, and YoY tests on the two multiyear clusters

In [ ]:
cluster_mechanisms = run_per_dataset_controlled_mechanisms(
    config,
    policy,
    dataset_keys=("CLUSTER_1", "CLUSTER_2"),
)
for key, table in cluster_mechanisms.items():
    print("\n", key)
    display(
        table.sort_values(["variant", "component", "seed"])
        .head(60)
    )


## 4. Structural and feature tests for the two BDG external-source checks

The BDG series are included for the structural/feature checks only. Year-over-year variants remain not applicable because each BDG aggregate contains one year of observations.


In [ ]:
bdg_parts = []

for dataset_key in ("BDG_EDU", "BDG_DORM"):
    settings = _read_existing_settings(config, dataset_key, policy)

    # Full-model reference from the frozen primary benchmark.
    benchmark_dir = core.dataset_output_dir(config, dataset_key)
    reference_rows = []
    for seed in policy.mechanism_seeds:
        for component, prediction_model in (
            ("SEQUENTIAL_ONLY_PRIMARY", "LASH_SEQ_COMPONENT"),
            ("REROUTED_PIPELINE_SECONDARY", "LASH"),
        ):
            payload = core.load_prediction(benchmark_dir, prediction_model, seed)
            reference_rows.append({
                "dataset": dataset_key,
                "variant": "FULL_MODEL_REFERENCE",
                "component": component,
                "seed": seed,
                **core.regression_metrics(payload["actual"], payload["prediction"]),
            })
    bdg_parts.append(pd.DataFrame(reference_rows))

    for name, flags in ablation.STRUCTURAL_VARIANTS.items():
        result = engine.run_one_frozen_variant(
            config, settings, dataset_key, name, policy,
            ablation_flags=flags,
        )
        bdg_parts.append(result["summary"])

    for name, options in ablation.FEATURE_VARIANTS.items():
        result = engine.run_one_frozen_variant(
            config, settings, dataset_key, name, policy,
            include_nonlinear_weather=options.get("include_nonlinear_weather", True),
            include_phase_shift_calendar=options.get("include_phase_shift_calendar", True),
        )
        bdg_parts.append(result["summary"])

bdg_summary = pd.concat(bdg_parts, ignore_index=True, sort=False)

reference = bdg_summary.loc[
    bdg_summary["variant"].eq("FULL_MODEL_REFERENCE"),
    ["dataset", "component", "seed", "selection_score"],
].rename(columns={"selection_score": "full_model_selection_score"})

bdg_summary = bdg_summary.merge(
    reference,
    on=["dataset", "component", "seed"],
    how="left",
)
bdg_summary["delta_selection_score_vs_full"] = (
    bdg_summary["selection_score"] - bdg_summary["full_model_selection_score"]
)

output = OUTPUT_DIR / "02_bdg_controlled_structural_feature_tests.xlsx"
with pd.ExcelWriter(output, engine="openpyxl") as writer:
    bdg_summary.to_excel(writer, sheet_name="Controlled_Effects", index=False)
    pd.DataFrame([{
        "datasets": "BDG_EDU, BDG_DORM",
        "annual_components": "N/A: one-year series",
        "primary_estimand": "SEQUENTIAL_ONLY_PRIMARY",
        "secondary_estimand": "REROUTED_PIPELINE_SECONDARY",
        "test_used_for_selection": False,
    }]).to_excel(writer, sheet_name="Protocol", index=False)

print("BDG structural/feature workbook:", output)
display(bdg_summary.head(80))


## 5. Output locations

In [ ]:
print("Cluster controlled mechanisms:", OUTPUT_DIR / "02_per_dataset_controlled_mechanism_tests.xlsx")
print("BDG structural/feature tests:", OUTPUT_DIR / "02_bdg_controlled_structural_feature_tests.xlsx")
print("Router sensitivity folders:", OUTPUT_DIR / "router_sensitivity")
print("Variant-level cached artifacts:", OUTPUT_DIR / "controlled_mechanism")
